# 07 — Regression

Mục tiêu: dự đoán `duration_minutes` từ các feature đã được tạo ở `05_feature_engineering.ipynb`.

Tiêu chí chọn model: ưu tiên **MAE thấp**, đồng thời xem RMSE và R².

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

data_path = '../data/processed/regression_data.csv'
df = pd.read_csv(data_path)

print("Kích thước dữ liệu:", df.shape)
display(df.head())

In [ ]:
X = df.drop(columns=['duration_minutes'])
y = df['duration_minutes']

print("Số lượng feature:", X.shape[1])
print("Features:")
print(X.columns.tolist())

print("\nTarget: duration_minutes")
print(y.describe().round(2))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    'Extra Trees': ExtraTreesRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
}

results = []
trained_models = {}

for name, model in models.items():
    print(f"Đang huấn luyện: {name}")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        'model': name,
        'mae': mae,
        'rmse': rmse,
        'r2': r2
    })

    trained_models[name] = model

    print(f"MAE : {mae:.2f} phút")
    print(f"RMSE: {rmse:.2f} phút")
    print(f"R²  : {r2:.4f}")
    print("-" * 50)

results_df = (
    pd.DataFrame(results)
    .sort_values(
        by=['mae', 'rmse'],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)

print("\nBảng so sánh Regression:")
display(results_df.round(4))

In [ ]:
best_model_name = results_df.loc[0, 'model']
best_regression_model = trained_models[best_model_name]

y_pred_best = best_regression_model.predict(X_test)

print("Model tốt nhất:", best_model_name)
print("MAE :", round(mean_absolute_error(y_test, y_pred_best), 2), "phút")
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_best)), 2), "phút")
print("R²  :", round(r2_score(y_test, y_pred_best), 4))

comparison = pd.DataFrame({
    'actual_minutes': y_test.to_numpy(),
    'predicted_minutes': np.round(y_pred_best, 2)
})

display(comparison.head(10))

In [ ]:
os.makedirs('../models/regression', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

model_path = '../models/regression/best_regression_model.pkl'
results_path = '../data/processed/regression_model_results.csv'

joblib.dump(best_regression_model, model_path)
results_df.to_csv(results_path, index=False)

print("Đã lưu model tốt nhất:")
print(model_path)

print("\nĐã lưu kết quả so sánh:")
print(results_path)

print("\nThứ tự feature của model:")
print(best_regression_model.feature_names_in_.tolist())

In [ ]:
sample = X_test.iloc[[0]]
prediction = best_regression_model.predict(sample)[0]

print("Thời gian thực tế :", round(float(y_test.iloc[0]), 2), "phút")
print("Thời gian dự đoán :", round(float(prediction), 2), "phút")